In [1]:
# 1 - CREATE - 1_REGISTER_ALL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '2_INPUT/riscv-unified-db/spec/std/isa/csr'
FOLDER_INPUT_OTHER = '2_INPUT/riscv-opcodes'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_BLOCK_SORT_KEY(STRING_BLOCK: str):
    MATCH = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    STRING_NAME = MATCH.group(1).strip() if MATCH else STRING_BLOCK
    return FUNCTION_NATURAL_SORT_KEY(STRING_NAME)

def FUNCTION_GET_PRIV_AND_WIDTH(STRING_REGISTER: str):
    if STRING_REGISTER.startswith(('seed', 'ssp')):
        return 'U-mode', 'XLEN'
    STRING_FIRST_CHAR = STRING_REGISTER[0].lower()
    if STRING_FIRST_CHAR == 'h':
        return 'HS-mode', 'SXLEN'
    if STRING_FIRST_CHAR == 'd':
        return 'D-mode', 'MXLEN'
    if STRING_FIRST_CHAR == 'm':
        return 'M-mode', 'MXLEN'
    if STRING_FIRST_CHAR == 's':
        return 'S-mode', 'SXLEN'
    return 'U-mode', 'XLEN'
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_OUTPUT = []
SET_REGISTER_YAML = set()
MAP_ENTRY_MISSING = {}
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_PATH, _, EACH_LIST_FILE in os.walk(FOLDER_INPUT):
    for EACH_FILE_NAME in EACH_LIST_FILE:
        if not EACH_FILE_NAME.endswith('.yaml'):
            continue
        FILE_PATH = os.path.join(EACH_PATH, EACH_FILE_NAME)
        with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            LIST_LINE = FILE_HANDLE.readlines()
        STRING_CURRENT = ''
        IS_FIELD = False
        LIST_EXTENSION = []
        LIST_BIT = []
        LIST_LOCATION = []
        INDENT_FIELDS = None
        IS_DEFINED_BY = False
        IS_ALL_OF = False
        HAD_FIELDS = False
        for EACH_LINE in LIST_LINE:
            STRING_LINE_STRIPPED = EACH_LINE.strip()
            if IS_FIELD:
                INDENT_LINE = len(EACH_LINE) - len(EACH_LINE.lstrip(' '))
                if INDENT_FIELDS is not None and INDENT_LINE <= INDENT_FIELDS and STRING_LINE_STRIPPED:
                    IS_FIELD = False
                if IS_FIELD:
                    if (
                        STRING_LINE_STRIPPED.endswith(':')
                        and not STRING_LINE_STRIPPED.startswith('-')
                        and ':' in STRING_LINE_STRIPPED
                        and INDENT_FIELDS is not None
                        and INDENT_LINE == INDENT_FIELDS + 2
                    ):
                        STRING_FIELD_NAME = STRING_LINE_STRIPPED[:-1].strip()
                        if STRING_FIELD_NAME not in ('fields', 'definedBy', 'allOf'):
                            LIST_BIT.append(STRING_FIELD_NAME)
                        continue
                    if STRING_LINE_STRIPPED.startswith(('location:', 'location_rv64:')):
                        LIST_LOCATION.append(EACH_LINE.split(':', 1)[1].strip())
                        continue
                    if STRING_LINE_STRIPPED.startswith(('affectedBy:', 'definedBy:')):
                        LIST_EXTENSION.append(EACH_LINE.split(':', 1)[1].strip())
                        continue
                    continue
            if STRING_LINE_STRIPPED.startswith('name:') and not IS_DEFINED_BY:
                STRING_REGISTER = EACH_LINE.split(':', 1)[1].strip()
                STRING_CURRENT += f'REGISTER: {STRING_REGISTER}\n'
            elif STRING_LINE_STRIPPED.startswith('address:'):
                STRING_CURRENT += f"NUMBER: {EACH_LINE.split(':', 1)[1].strip().lower()}\n"
            elif STRING_LINE_STRIPPED.startswith('priv_mode:'):
                STRING_PRIV = EACH_LINE.split(':', 1)[1].strip()
                if STRING_CURRENT.startswith('REGISTER: h') and STRING_PRIV == 'S':
                    STRING_PRIV = 'HS'
                STRING_CURRENT += f'MODE: {STRING_PRIV}-mode\n'
            elif STRING_LINE_STRIPPED.startswith('length:'):
                STRING_CURRENT += f"WIDTH: {EACH_LINE.split(':', 1)[1].strip()}\n"
            elif STRING_LINE_STRIPPED.startswith('definedBy:'):
                LIST_EXTENSION.append(EACH_LINE.split(':', 1)[1].strip())
                IS_DEFINED_BY = True
            elif IS_DEFINED_BY and not STRING_LINE_STRIPPED.startswith('fields:'):
                if STRING_LINE_STRIPPED.startswith('allOf:'):
                    IS_ALL_OF = True
                elif IS_ALL_OF and STRING_LINE_STRIPPED.startswith('-'):
                    LIST_EXTENSION.append(EACH_LINE.split('-', 1)[1].strip())
                elif STRING_LINE_STRIPPED.startswith('name:'):
                    LIST_EXTENSION.append(EACH_LINE.split('name:', 1)[1].strip())
            elif STRING_LINE_STRIPPED.startswith('fields:'):
                IS_ALL_OF = False
                IS_DEFINED_BY = False
                if STRING_LINE_STRIPPED == 'fields: {}':
                    LIST_EXTENSION = [EACH_EXTENSION for EACH_EXTENSION in LIST_EXTENSION if EACH_EXTENSION]
                    if not LIST_EXTENSION:
                        LIST_EXTENSION.append('-')
                    STRING_CURRENT += 'FIELDS: -\nEXTENSIONS: -'
                    break
                IS_FIELD = True
                HAD_FIELDS = True
                INDENT_FIELDS = len(EACH_LINE) - len(EACH_LINE.lstrip(' '))
        if HAD_FIELDS:
            LIST_BIT_FINAL = []
            if LIST_LOCATION and LIST_LOCATION[0].startswith('0'):
                ITER_PAIR = zip(reversed(LIST_LOCATION), reversed(LIST_BIT))
                for EACH_LOCATION, EACH_NAME in ITER_PAIR:
                    if '-' in EACH_LOCATION:
                        STRING_A, STRING_B = EACH_LOCATION.split('-')
                        LIST_BIT_FINAL.append(f'{STRING_B}..{STRING_A}={EACH_NAME}')
                    else:
                        LIST_BIT_FINAL.append(f'{EACH_LOCATION}..{EACH_LOCATION}={EACH_NAME}')
            else:
                ITER_PAIR = zip(LIST_LOCATION, LIST_BIT)
                for EACH_LOCATION, EACH_NAME in ITER_PAIR:
                    if '-' in EACH_LOCATION:
                        STRING_A, STRING_B = EACH_LOCATION.split('-')
                        LIST_BIT_FINAL.append(f'{STRING_A}..{STRING_B}={EACH_NAME}')
                    else:
                        LIST_BIT_FINAL.append(f'{EACH_LOCATION}..{EACH_LOCATION}={EACH_NAME}')
            if LIST_BIT_FINAL:
                STRING_CURRENT += f"FIELDS: {', '.join(LIST_BIT_FINAL)}\n"
            else:
                STRING_CURRENT += 'FIELDS: -\n'
            if LIST_EXTENSION:
                LIST_EXTENSION = [EACH_EXTENSION for EACH_EXTENSION in LIST_EXTENSION if EACH_EXTENSION]
                LIST_EXTENSION_FLAT = []
                for EACH_EXTENSION in LIST_EXTENSION:
                    if EACH_EXTENSION.startswith('[') and EACH_EXTENSION.endswith(']'):
                        LIST_EXTENSION_FLAT.extend(EACH_PART.strip() for EACH_PART in EACH_EXTENSION[1:-1].split(','))
                    else:
                        LIST_EXTENSION_FLAT.append(EACH_EXTENSION)
                LIST_EXTENSION = sorted(set(EACH_X.split('#')[0].strip() for EACH_X in LIST_EXTENSION_FLAT))
                STRING_CURRENT += 'EXTENSIONS: -'
            else:
                STRING_CURRENT += 'EXTENSIONS: -'
        if STRING_CURRENT:
            if re.search(r'(?m)^\s*MODE:\s*D-mode\s*$', STRING_CURRENT):
                continue
            LIST_OUTPUT.append(STRING_CURRENT)
            for EACH_LINE in STRING_CURRENT.splitlines():
                if EACH_LINE.startswith('REGISTER:'):
                    SET_REGISTER_YAML.add(EACH_LINE.split('REGISTER:', 1)[1].strip())

LIST_FILE_CSV = [EACH_FILE_NAME for EACH_FILE_NAME in os.listdir(FOLDER_INPUT_OTHER) if EACH_FILE_NAME.startswith('csrs') and EACH_FILE_NAME.endswith('.csv')]

for EACH_FILE_NAME in sorted(LIST_FILE_CSV, key=FUNCTION_NATURAL_SORT_KEY):
    FILE_PATH = os.path.join(FOLDER_INPUT_OTHER, EACH_FILE_NAME)
    with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        for EACH_LINE in FILE_HANDLE:
            STRING_LINE = EACH_LINE.strip()
            if not STRING_LINE or STRING_LINE.startswith('#'):
                continue
            LIST_PART = STRING_LINE.split(',')
            if len(LIST_PART) < 2:
                continue
            STRING_HEX = LIST_PART[0].strip().lower()
            STRING_REGISTER = LIST_PART[1].strip().strip('"')
            if STRING_REGISTER not in SET_REGISTER_YAML:
                STRING_PRIV, STRING_WIDTH = FUNCTION_GET_PRIV_AND_WIDTH(STRING_REGISTER)
                if STRING_PRIV == 'D-mode':
                    continue
                MAP_ENTRY_MISSING[STRING_REGISTER] = {'hex': STRING_HEX, 'priv': STRING_PRIV, 'width': STRING_WIDTH}

for EACH_REGISTER in list(MAP_ENTRY_MISSING):
    if EACH_REGISTER.endswith('h'):
        STRING_BASE = EACH_REGISTER[:-1]
        if STRING_BASE in MAP_ENTRY_MISSING:
            MAP_ENTRY_MISSING[STRING_BASE]['width'] = '64'
            MAP_ENTRY_MISSING[EACH_REGISTER]['width'] = '32'

for EACH_REGISTER, EACH_INFO in sorted(MAP_ENTRY_MISSING.items()):
    LIST_OUTPUT.append(f'REGISTER: {EACH_REGISTER}\n' f"NUMBER: {EACH_INFO['hex']}\n" f"MODE: {EACH_INFO['priv']}\n" f"WIDTH: {EACH_INFO['width']}\n" f'FIELDS: -\n' f'EXTENSIONS: -')

LIST_OUTPUT.sort(key=FUNCTION_BLOCK_SORT_KEY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_REGISTER_ALL.txt - END

0.26 second(s)
====
DONE
====


In [2]:
# 2 - CREATE - 2_REGISTER_SOME.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_FIND_V_ALIAS(TEXT_VALUE: str):
    LIST_NAME = [EACH_MATCH.group(1).strip() for EACH_MATCH in re.finditer(r'^REGISTER:\s*([^\n]+)', TEXT_VALUE, flags=re.M)]
    SET_NAME = set(LIST_NAME)
    SET_SEEN = set()
    LIST_OUTPUT = []
    for EACH_NAME in LIST_NAME:
        if (EACH_NAME.startswith('v') and len(EACH_NAME) > 1 and EACH_NAME[1:] in SET_NAME and EACH_NAME not in SET_SEEN):
            LIST_OUTPUT.append(EACH_NAME)
            SET_SEEN.add(EACH_NAME)
    return LIST_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_NOT_IMPLEMENTED = [
    'sireg', 'sireg2', 'sireg3', 'sireg4', 'sireg5', 'sireg6',
    'vsireg', 'vsireg2', 'vsireg3', 'vsireg4', 'vsireg5', 'vsireg6',
    'jvt', 'mnstatus', 'mncause', 'tcontrol',
]
LIST_NOT_IMPLEMENTED += [
    f'hpmcounter{i}' for i in range(19, 32)
] + [
    f'hpmcounter{i}h' for i in range(19, 32)
]
LIST_NOT_IMPLEMENTED += [
    f'mhpmcounter{i}' for i in range(19, 32)
] + [
    f'mhpmcounter{i}h' for i in range(19, 32)
]
LIST_REGISTER_WANTED = ['hpmcounter18', 'mcause', 'mcounteren', 'mcountinhibit', 'mcycle', 'medeleg', 'menvcfg', 'mepc', 'mhartid', 'mhpmcounter18', 'mhpmevent18', 'mideleg', 'minstret', 'mscratch', 'mtvec', 'pmpaddr0', 'pmpcfg0', 'satp', 'scause', 'scounteren', 'scountinhibit', 'sepc', 'sstatus', 'stvec', 'tdata1', 'tselect', 'fflags', 'frm', 'fcsr', 'vstart', 'vl', 'vtype', 'vlenb', 'vxrm', 'vxsat', 'vcsr']
IS_USING_EVERYTHING = False
# VARIABLE - FINISH

# CODE - BEGIN
LIST_REGISTER_WANTED = [
    EACH_REGISTER
    for EACH_REGISTER in LIST_REGISTER_WANTED
    if EACH_REGISTER not in LIST_NOT_IMPLEMENTED
]

if not LIST_REGISTER_WANTED:
    LIST_REGISTER_WANTED.append('tdata1')

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_ALL = FILE_HANDLE.read()

SET_V_ALIAS = set(FUNCTION_FIND_V_ALIAS(TEXT_ALL))

for EACH_V_ALIAS in (SET_V_ALIAS & set(LIST_REGISTER_WANTED)):
    LIST_REGISTER_WANTED.append(EACH_V_ALIAS[1:])

LIST_REGISTER_WANTED.sort(key=FUNCTION_NATURAL_SORT_KEY)

LIST_LINE = TEXT_ALL.splitlines(True)
SET_NOT_IMPLEMENTED = {EACH.lower() for EACH in LIST_NOT_IMPLEMENTED}

if IS_USING_EVERYTHING:
    LIST_OUT = []
    EACH_INDEX_LINE = 0
    while EACH_INDEX_LINE < len(LIST_LINE):
        EACH_LINE = LIST_LINE[EACH_INDEX_LINE]
        STRIPPED = EACH_LINE.strip()
        if STRIPPED.startswith('REGISTER:'):
            NAME = STRIPPED.split(':', 1)[1].strip().lower()
            if NAME in SET_NOT_IMPLEMENTED:
                EACH_INDEX_LINE += 1
                while (
                    EACH_INDEX_LINE < len(LIST_LINE)
                    and not LIST_LINE[EACH_INDEX_LINE].strip().startswith('REGISTER:')
                ):
                    EACH_INDEX_LINE += 1
                continue
        LIST_OUT.append(EACH_LINE)
        EACH_INDEX_LINE += 1
    TEXT_FINAL = ''.join(LIST_OUT)
    TEXT_FINAL = re.sub(r'\n{3,}', '\n\n', TEXT_FINAL).strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_FINAL)
else:
    LIST_WANTED = sorted({EACH_WANTED.lower() for EACH_WANTED in LIST_REGISTER_WANTED})
    LIST_SELECTED = []
    for EACH_WANTED in LIST_WANTED:
        STRING_TARGET = f'REGISTER: {EACH_WANTED}'
        for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
            if EACH_LINE.strip() == STRING_TARGET:
                LIST_SELECTED.append(''.join(EACH_L.strip() + '\n' for EACH_L in LIST_LINE[EACH_INDEX_LINE:EACH_INDEX_LINE + 6]).rstrip('\n'))
                break
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write('\n\n'.join(LIST_SELECTED))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_REGISTER_SOME.txt - END

0.00 second(s)
====
DONE
====


In [3]:
# 3 - CREATE - 3_REGISTER_BOOLEAN.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '3_SPECIFICATION/2_SPLIT'
FILE_INPUT = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_PROMPT = '5_REGISTER/2_HISTORY/1_HISTORY_REGISTER_BOOLEAN.txt'
FILE_ALL = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_LLM_CALL(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 16384
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
STRING_PREFIX_REGISTER_DIGIT = ''
LIST_OUTPUT_LINE = []
LIST_HISTORY = []
GLOBAL_COUNTER = 1
SPECIAL_NAME_FOR_NUMBERS = ''
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_REGISTER = [EACH_LINE.split(': ', 1)[1].strip() for EACH_LINE in FILE_HANDLE if EACH_LINE.startswith('REGISTER: ')]

with open(FILE_ALL, encoding='utf-8') as FILE_HANDLE:
    LIST_ALL_LINE = FILE_HANDLE.readlines()

for EACH_REGISTER in LIST_REGISTER:
    LIST_WORD = [EACH_REGISTER]
    if re.search(r'\d', EACH_REGISTER):
        MATCH_PREFIX = re.match(r'^[a-zA-Z]+', EACH_REGISTER)
        if MATCH_PREFIX:
            SPECIAL_NAME_FOR_NUMBERS = MATCH_PREFIX.group(0)
            LIST_WORD.append(SPECIAL_NAME_FOR_NUMBERS)
    elif EACH_REGISTER.endswith('h'):
        STRING_BASE = EACH_REGISTER[:-1]
        if STRING_BASE and STRING_BASE != EACH_REGISTER:
            LIST_WORD.append(STRING_BASE)
    elif EACH_REGISTER.startswith('v'):
        STRING_BASE = EACH_REGISTER[1:]
        if (STRING_BASE and STRING_BASE != EACH_REGISTER and any(f'REGISTER: {STRING_BASE}' in EACH_LINE for EACH_LINE in LIST_ALL_LINE)):
            LIST_WORD.append(STRING_BASE)
    LIST_WORD = list(dict.fromkeys(LIST_WORD))
    LIST_PATH = []
    for EACH_FILE_NAME in sorted(os.listdir(FOLDER_INPUT), key=FUNCTION_NATURAL_SORT_KEY):
        if not EACH_FILE_NAME.endswith('.txt'):
            continue
        FILE_PATH = os.path.join(FOLDER_INPUT, EACH_FILE_NAME)
        with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            if any(any(EACH_WORD in EACH_LINE for EACH_WORD in LIST_WORD) for EACH_LINE in FILE_HANDLE):
                LIST_PATH.append(FILE_PATH)
    LIST_PATH.sort(key=lambda EACH_PATH: int(EACH_PATH.split('SPECIFICATION_')[1].split('.txt')[0]))
    LOCAL_COUNTER = 1
    for EACH_FILE_PATH in LIST_PATH:
        with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            STRING_SPEC = FILE_HANDLE.read()
        STRING_PREFIX_REGISTER = (EACH_REGISTER if STRING_PREFIX_REGISTER_DIGIT == '' else STRING_PREFIX_REGISTER_DIGIT)
        if SPECIAL_NAME_FOR_NUMBERS == '':
            SPECIAL_NAME_FOR_NUMBERS_PROMPT = ''
        else:
            SPECIAL_NAME_FOR_NUMBERS_PROMPT = f' or {SPECIAL_NAME_FOR_NUMBERS.strip()}'
        STRING_PROMPT = f'''For RISC-V, does this text below contain any information about {STRING_PREFIX_REGISTER}{SPECIAL_NAME_FOR_NUMBERS_PROMPT} register? Output YES or NO only.

The text:

{STRING_SPEC}'''
        STRING_RESPONSE = FUNCTION_CLEAN_TEXT(FUNCTION_LLM_CALL(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT))
        STRING_INTRO_FIRST = f' START - {GLOBAL_COUNTER} '.center(WIDTH, '=')
        STRING_INTRO_END = f' END - {GLOBAL_COUNTER} '.center(WIDTH, '=')
        STRING_INTRO = f'''GLOBAL NUMBER: {GLOBAL_COUNTER}
LOCAL NUMBER: {LOCAL_COUNTER}
NAME: {STRING_PREFIX_REGISTER}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
        STRING_ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
        LIST_HISTORY.append(STRING_INTRO_FIRST + '\n' + STRING_INTRO + STRING_PROMPT + STRING_ENDING + STRING_INTRO_END + SUFFIX)
        if 'YES' in STRING_RESPONSE:
            LIST_OUTPUT_LINE.append(f'{EACH_REGISTER} - {EACH_FILE_PATH}')
        GLOBAL_COUNTER += 1
        LOCAL_COUNTER += 1

STRING_PROMPT_HISTORY = ''.join(LIST_HISTORY).removesuffix(SUFFIX)

with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n'.join(LIST_OUTPUT_LINE))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 3 - CREATE - 3_REGISTER_BOOLEAN.txt - END

1 minute(s) 18.51 second(s)
====
DONE
====


In [4]:
# 4 - CREATE - 4_REGISTER_RULE.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_PROMPT_1 = '5_REGISTER/2_HISTORY/2_HISTORY_REGISTER_RULE_1.txt'
FILE_PROMPT_2 = '5_REGISTER/2_HISTORY/3_HISTORY_REGISTER_RULE_2.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_LLM(STRING_PROMPT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_NORMALIZE_RULE(TEXT_VALUE):
    LIST_LINE = [EACH_LN.strip() for EACH_LN in TEXT_VALUE.splitlines() if EACH_LN.strip()]
    LIST_RULE = []
    for EACH_LN in LIST_LINE:
        if re.match(r'^RULE\s+\d+:', EACH_LN, re.I):
            EACH_LN = re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LN)
            LIST_RULE.append(EACH_LN)
    LIST_FIXED = []
    for EACH_LN in LIST_RULE:
        MATCH_RULE = re.match(r'^\s*RULE\s+(\d+):\s*(.*)', EACH_LN)
        if not MATCH_RULE:
            LIST_FIXED.append(EACH_LN)
            continue
        NUMBER_RULE = MATCH_RULE.group(1)
        STRING_REST = MATCH_RULE.group(2)
        STRING_REST = re.sub(rf'^RULE\s+{NUMBER_RULE}:\s*', '', STRING_REST)
        LIST_FIXED.append(f'RULE {NUMBER_RULE}: {STRING_REST}')
    return '\n'.join(LIST_FIXED)
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_BLOCK_OUTPUT = []
MAP_REGISTER_RULE = defaultdict(list)
LIST_HISTORY_1 = []
LIST_HISTORY_2 = []
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
REGISTER_INITIAL = None
LIST_BLOCK_FINAL = []
LIST_SUPER_FINAL = []
IS_64_BIT = True
PATTERN_RV32 = re.compile(r'\bRV32\b')
PATTERN_32BIT = re.compile(r'\b32[\-\u2011]bits?\b')
PATTERN_WFI = re.compile(r'\bWFI\b')
PATTERN_INTERRUPT = re.compile(r'\binterrupts\b', re.IGNORECASE)
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

for EACH_REGISTER in LIST_REGISTER_ORDER:
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER]).strip()
    LOCAL_COUNTER_1 = 1
    LOCAL_COUNTER_2 = 1
    
    # PROMPT 1 - BEGIN
    STRING_PROMPT_1 = f'''For RISC-V, please extract all possible information about {EACH_REGISTER} register from the text below. The information must be only about {EACH_REGISTER} register, not other things. Extract ONLY explicit, testable, normative description about architectural behavior.

Please output in paragraphs. If you want, you can include bullet or numbered points in your output. You can also include tables. However, do not give me any introduction in your output. You just need to output the information that you extract.

If your output includes some sentences that refer to some tables or figures, include those tables or figures too.

Remember, output must be ASCII-only text.

If it happens to be there is no information about {EACH_REGISTER} register from the text below, output NONE

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Your description must be very clear and unambiguous. If you want, you can output in bullet points also.

Use the exact terminology and wording used in the text below; do not substitute synonyms or paraphrase architectural terms.

The text below:

{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_LLM(STRING_PROMPT_1)
    if STRING_RESPONSE_1.strip() == 'NONE':
        continue
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(STRING_RESPONSE_1)
    LIST_HISTORY_1.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_1, LOCAL_COUNTER_1, EACH_REGISTER, STRING_PROMPT_1, STRING_RESPONSE_1))
    # PROMPT 1 - FINISH

    # PROMPT 2 - BEGIN
    STRING_PROMPT_2 = f'''For RISC-V, for {EACH_REGISTER} register, rewrite the following description as a list of rules. Each rule must explain about {EACH_REGISTER} register. Essentially, for each rule, it is about {EACH_REGISTER} register, not other things.

Do not lose any information about {EACH_REGISTER} register from the description. Read the text or the description carefully. Understand the context and the sentences. Do not lose any important information.

For each rule, do not use any internal references like "According to Table 7," or "see Figure 3." Each rule must be one sentence only. Each rule must stand fully on its own without cross-references. Do not quote or refer to other rules. Essentially, each rule is one idea. Avoid a rule like this: "The register being described is {EACH_REGISTER}."

Each rule should become a single testable statement. Each rule should be a separate atomic rule, each expressing only one fact. Each rule needs to be clear and detailed. Extract ONLY explicit, testable, normative rules about architectural behavior. A rule MUST describe behavioral semantics, not naming, encoding, or syntax.

Remember, rules must describe architectural cause -> effect semantics (event -> architectural outcome) rather than decoding the meaning of an already-set register value.

Remember, if a sentence contains multiple distinct behaviors or enumerated cases, split them into multiple rules so each rule expresses only one case.

Do not write rules in the form "{EACH_REGISTER} has <bit pattern> means/represents/indicates <cause>"; instead, write them as "If <cause/event> occurs, {EACH_REGISTER} becomes/contains <bit pattern>".

Exclude purely structural or descriptive properties (such as register width, naming, or existence) unless they imply observable architectural behavior.

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Ignore any statements that describe reset values, power-on defaults, or initial state after reset; do not generate rules from reset-related information.

OUTPUT FORMAT:
+ Each rule must begin with the word "RULE", followed by its number (e.g., "RULE 1: ", "RULE 2: ", "RULE 3: ", ...).
+ Do not give me any introduction.
+ Each rule must contain the word {EACH_REGISTER}.

Remember, output must be ASCII-only text.

The following description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_2))
    LIST_HISTORY_2.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_2, LOCAL_COUNTER_2, EACH_REGISTER, STRING_PROMPT_2, STRING_RESPONSE_2))
    # PROMPT 2 - FINISH

    # FINALIZING - BEGIN
    STRING_RESPONSE_FINISH = STRING_RESPONSE_2
    STRING_CLEANED = FUNCTION_NORMALIZE_RULE(STRING_RESPONSE_FINISH)
    LIST_BLOCK_OUTPUT.append(f'REGISTER: {EACH_REGISTER}\n{STRING_CLEANED}')
    # FINALIZING - FINISH

    # CLEANING UP PROMPT - BEGIN
    GLOBAL_COUNTER_1 += 1
    LOCAL_COUNTER_1 += 1
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_1).removesuffix(SUFFIX))

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_2).removesuffix(SUFFIX))
# CLEANING UP PROMPT - FINISH

for EACH_BLOCK in LIST_BLOCK_OUTPUT:
    LIST_LINE = EACH_BLOCK.strip().splitlines()
    
    # BONUS FEATURE - DELETE UNWORTHY RULES - BEGIN
    if IS_64_BIT:
        LIST_LINE = [
            EACH_LINE
            for EACH_LINE in LIST_LINE
            if not (
                PATTERN_RV32.search(EACH_LINE)
                or PATTERN_32BIT.search(EACH_LINE)
                or PATTERN_WFI.search(EACH_LINE)
                or PATTERN_INTERRUPT.search(EACH_LINE)
                or 'XLEN=32' in EACH_LINE
                or 'XLEN = 32' in EACH_LINE
                or 'XLEN is 32' in EACH_LINE
                or 'XLEN equals 32' in EACH_LINE
            )
        ]
    # BONUS FEATURE - DELETE UNWORTHY RULES - FINISH
    
    STRING_NAME = LIST_LINE[0].split(':', 1)[1].strip()
    MAP_REGISTER_RULE[STRING_NAME].extend(LIST_LINE[1:])

for EACH_NAME, EACH_LIST_RULE in MAP_REGISTER_RULE.items():
    LIST_LINE_NEW = [f'REGISTER: {EACH_NAME}']
    for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(EACH_LIST_RULE, 1):
        LIST_LINE_NEW.append(f"RULE {EACH_INDEX_RULE}: {EACH_RULE_LINE.split(':', 1)[1].strip()}")
    LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE_NEW))

for EACH_BLOCK in LIST_BLOCK_FINAL:
    LIST_LINE = EACH_BLOCK.splitlines()
    STRING_HEADER = LIST_LINE[0].strip()
    LIST_RULE = [EACH_R.strip() for EACH_R in LIST_LINE[1:] if EACH_R.strip() and EACH_R.strip() != 'RULE:']
    LIST_RULE = [re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_R) for EACH_R in LIST_RULE]
    LIST_RULE = [re.sub(r'RULE\s+\d+:', f'RULE {EACH_I}:', EACH_R) for EACH_I, EACH_R in enumerate(LIST_RULE, 1)]
    LIST_SUPER_FINAL.append(STRING_HEADER + '\n' + '\n'.join(LIST_RULE))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_SUPER_FINAL))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 4 - CREATE - 4_REGISTER_RULE.txt - END

2 minute(s) 5.72 second(s)
====
DONE
====


In [5]:
# 5 - CREATE - 5_REGISTER_EXTENSION.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_PROMPT = '5_REGISTER/2_HISTORY/4_HISTORY_REGISTER_EXTENSION.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_LLM(STRING_PROMPT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_SPLIT_BLOCK(TEXT_VALUE: str):
    LIST_BLOCK = []
    LIST_CURRENT = []
    for EACH_LINE in TEXT_VALUE.splitlines(True):
        if EACH_LINE.startswith('REGISTER:') and LIST_CURRENT:
            LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
            LIST_CURRENT = []
        LIST_CURRENT.append(EACH_LINE)
    if LIST_CURRENT:
        LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
    return LIST_BLOCK

def FUNCTION_TOKEN_OK(STRING_TOKEN: str):
    STRING_TOKEN = STRING_TOKEN.strip()
    if not STRING_TOKEN or ' ' in STRING_TOKEN or not STRING_TOKEN.isalpha():
        return False
    return STRING_TOKEN[0].isupper() and (len(STRING_TOKEN) == 1 or STRING_TOKEN[1:].islower())
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_HISTORY = []
GLOBAL_COUNTER = 1
LOCAL_COUNTER = 1
REGISTER_INITIAL = None
MAP_MERGED = {}
LIST_OUTPUT_ENTRY = []
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

for EACH_REGISTER in LIST_REGISTER_ORDER:
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER]).strip()
    LOCAL_COUNTER = 1
    STRING_PROMPT = f'''For RISC-V, based on the text below, output all extensions (RISC-V extensions) for {EACH_REGISTER} register. Note: registers, CSRs, fields, modes, interrupts, and instructions are not extensions.

Read the text below carefully and make sure you don't miss any extensions!

HOW TO OUTPUT:
Enter extensions as a comma-separated list on one line. If there are no extensions for {EACH_REGISTER} register in the rules, output NONE.

The text below:

{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT))
    LIST_HISTORY.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER, LOCAL_COUNTER, EACH_REGISTER, STRING_PROMPT, STRING_RESPONSE))
    STRING_RESPONSE_2 = STRING_RESPONSE.replace(EACH_REGISTER, '')
    if 'NONE' not in STRING_RESPONSE_2:
        if EACH_REGISTER in MAP_MERGED:
            MAP_MERGED[EACH_REGISTER] = f'{MAP_MERGED[EACH_REGISTER]}, {STRING_RESPONSE_2}'
        else:
            MAP_MERGED[EACH_REGISTER] = STRING_RESPONSE_2
    GLOBAL_COUNTER += 1
    LOCAL_COUNTER += 1

with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY).removesuffix(SUFFIX))

for EACH_REGISTER in sorted(MAP_MERGED, key=str.lower):
    SET_SEEN = set()
    LIST_TOKEN = []
    for EACH_TOKEN in MAP_MERGED[EACH_REGISTER].split(','):
        STRING_TOKEN = EACH_TOKEN.strip()
        if STRING_TOKEN and STRING_TOKEN not in SET_SEEN:
            SET_SEEN.add(STRING_TOKEN)
            LIST_TOKEN.append(STRING_TOKEN)
    LIST_TOKEN = [EACH_TOKEN for EACH_TOKEN in LIST_TOKEN if FUNCTION_TOKEN_OK(EACH_TOKEN)]
    LIST_OUTPUT_ENTRY.append(f"REGISTER: {EACH_REGISTER}\nEXTENSIONS: {', '.join(LIST_TOKEN)}")

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_ENTRY))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 5 - CREATE - 5_REGISTER_EXTENSION.txt - END

22.64 second(s)
====
DONE
====


In [6]:
# 6 - CREATE - 6_REGISTER_FIELD.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_PROMPT_1 = '5_REGISTER/2_HISTORY/5_HISTORY_REGISTER_FIELD_1.txt'
FILE_PROMPT_2 = '5_REGISTER/2_HISTORY/6_HISTORY_REGISTER_FIELD_2.txt'
FILE_OTHER_WIDTH = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_OTHER_INPUT = '5_REGISTER/1_OUTPUT/3_REGISTER_BOOLEAN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_LLM(STRING_PROMPT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_HISTORY_BLOCK(GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_SPLIT_BLOCK(LIST_LINE):
    LIST_BLOCK = []
    LIST_CURRENT = []
    for EACH_LINE in LIST_LINE:
        if EACH_LINE.startswith('REGISTER:') and LIST_CURRENT:
            LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
            LIST_CURRENT = []
        LIST_CURRENT.append(EACH_LINE)
    if LIST_CURRENT:
        LIST_BLOCK.append(''.join(LIST_CURRENT).strip())
    return LIST_BLOCK
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
LIST_REGISTER_NAME = []
MAP_WIDTH = {}
LIST_INJECTED = []
LIST_OUTPUT_BLOCK = []
LIST_HISTORY_1 = []
LIST_HISTORY_2 = []
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
REGISTER_INITIAL = None
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_OTHER_WIDTH, encoding='utf-8') as FILE_HANDLE:
    STRING_REGISTER = None
    STRING_WIDTH = None
    for EACH_LINE in FILE_HANDLE:
        if EACH_LINE.startswith('REGISTER: '):
            STRING_REGISTER = EACH_LINE.split(': ', 1)[1].strip()
        elif EACH_LINE.startswith('WIDTH: '):
            STRING_WIDTH = EACH_LINE.split(': ', 1)[1].strip()
        elif (EACH_LINE.startswith('FIELDS: -') and STRING_REGISTER is not None and STRING_WIDTH is not None):
            STRING_PAIR = f'{STRING_REGISTER} - {STRING_WIDTH}'
            LIST_REGISTER_NAME.append(STRING_PAIR)
            MAP_WIDTH[STRING_REGISTER] = STRING_WIDTH

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_RULE_LINE = FILE_HANDLE.readlines()

for EACH_LINE in LIST_RULE_LINE:
    if EACH_LINE.startswith('REGISTER:'):
        LIST_INJECTED.append(EACH_LINE)
        STRING_REGISTER = EACH_LINE.split('REGISTER:', 1)[1].strip()
        if STRING_REGISTER in MAP_WIDTH:
            LIST_INJECTED.append(f'WIDTH: {MAP_WIDTH[STRING_REGISTER]}\n')

LIST_INPUT_2 = FUNCTION_SPLIT_BLOCK(LIST_INJECTED)
MAP_BLOCK_BY_REGISTER = {EACH_BLOCK.splitlines()[0].split('REGISTER:', 1)[1].strip(): EACH_BLOCK for EACH_BLOCK in LIST_INPUT_2}

with open(FILE_OTHER_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_REGISTER, EACH_FILE_PATH in LIST_PAIR:
    if EACH_REGISTER not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_REGISTER)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_REGISTER].append(FILE_HANDLE.read())

for EACH_INDEX_SPECIAL, EACH_SPECIAL in enumerate(LIST_REGISTER_NAME):
    STRING_REGISTER, STRING_WIDTH = (EACH_PART.strip() for EACH_PART in EACH_SPECIAL.split(' - ', 1))
    STRING_BLOCK = MAP_BLOCK_BY_REGISTER.get(STRING_REGISTER, '')
    if EACH_INDEX_SPECIAL == 0:
        REGISTER_INITIAL = STRING_REGISTER
    elif STRING_REGISTER != REGISTER_INITIAL:
        REGISTER_INITIAL = STRING_REGISTER
        LOCAL_COUNTER_1 = 1
        LOCAL_COUNTER_2 = 1
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[STRING_REGISTER]).strip()
    STRING_PROMPT_1 = f'''For RISC-V, based on the text below, output all fields for {STRING_REGISTER} register. Do not give any introduction.

The text below:

{STRING_BLOCK}
TEXT:
{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_1))
    LIST_HISTORY_1.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_1, LOCAL_COUNTER_1, STRING_REGISTER, STRING_PROMPT_1, STRING_RESPONSE_1))
    NUMBER_WIDTH = int(STRING_WIDTH) if STRING_WIDTH.isdigit() else 64
    STRING_PROMPT_2 = f'''For RISC-V, from the description below, extract the fields for {STRING_REGISTER} register. The register of {STRING_REGISTER} is a {NUMBER_WIDTH}-bit register.

OUTPUT FORMAT:
FIELDS: <high_bit>..<low_bit>=<FIELD_NAME>, <high_bit>..<low_bit>=<FIELD_NAME>, ...

RULES:
+ Range Delimiter -> Always use two dots (..) for inclusive ranges (e.g., 7..0). Repeat the index for single-bit fields (e.g., 3..3).
+ Coverage -> Define every bit from {NUMBER_WIDTH - 1} down to 0, in contiguous, non-overlapping ranges. No gaps.
+ Reserved Bits -> If the description says something like "unused", "future use", or "reserved", then the <FIELD_NAME> must be WPRI.
+ Unknown Positions -> If a field's bit range is not given in the description, append the field at the end. The <high_bit> and the <low_bit> become "UNKNOWN". For example, "UNKNOWN..UNKNOWN=<FIELD_NAME>".
+ Ordering -> Based on the position or the bit range, list fields strictly from high to low, separated by ", " (comma and space).
+ No Overlaps -> Ensure bit ranges don't overlap.

Do not output any introduction.

The description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_LLM(STRING_PROMPT_2))
    LIST_HISTORY_2.append(FUNCTION_HISTORY_BLOCK(GLOBAL_COUNTER_2, LOCAL_COUNTER_2, STRING_REGISTER, STRING_PROMPT_2, STRING_RESPONSE_2))
    LIST_OUTPUT_BLOCK.append(f'REGISTER: {STRING_REGISTER}\n{STRING_RESPONSE_2}')
    GLOBAL_COUNTER_1 += 1
    LOCAL_COUNTER_1 += 1
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_1).removesuffix(SUFFIX))

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_HISTORY_2).removesuffix(SUFFIX))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_BLOCK))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 6 - CREATE - 6_REGISTER_FIELD.txt - END

15.62 second(s)
====
DONE
====


In [7]:
# 7 - CREATE - 7_REGISTER_FINAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_LIST = '5_REGISTER/1_OUTPUT/2_REGISTER_SOME.txt'
FILE_RULE = '5_REGISTER/1_OUTPUT/4_REGISTER_RULE.txt'
FILE_EXTENSION = '5_REGISTER/1_OUTPUT/5_REGISTER_EXTENSION.txt'
FILE_BIT = '5_REGISTER/1_OUTPUT/6_REGISTER_FIELD.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_READ(FILE_PATH):
    PATH_VALUE = Path(FILE_PATH)
    return PATH_VALUE.read_text(encoding='utf-8') if PATH_VALUE.exists() else ''

def FUNCTION_SPLIT_SECTION(TEXT_VALUE: str):
    MAP_OUTPUT = {}
    for EACH_PART in re.split(r'(?=^REGISTER:\s*)', TEXT_VALUE, flags=re.M):
        if not EACH_PART.strip():
            continue
        MATCH_REGISTER = re.match(r'^REGISTER:\s*(.+?)\s*[\r\n]+', EACH_PART)
        if MATCH_REGISTER:
            MAP_OUTPUT[MATCH_REGISTER.group(1).strip()] = EACH_PART.strip()
    return MAP_OUTPUT
# FUNCTION - FINISH

# VARIABLE - BEGIN
MAP_SECTION_BASE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_LIST))
MAP_SECTION_FIELD = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_BIT))
MAP_SECTION_EXTENSION = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_EXTENSION))
MAP_SECTION_RULE = FUNCTION_SPLIT_SECTION(FUNCTION_READ(FILE_RULE))
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_REGISTER, EACH_BASE_SECTION in MAP_SECTION_BASE.items():
    LIST_LINE = EACH_BASE_SECTION.splitlines()
    if EACH_REGISTER in MAP_SECTION_FIELD:
        MATCH_FIELD = re.search(r'^\s*FIELDS:\s*(.+)$', MAP_SECTION_FIELD[EACH_REGISTER], flags=re.M)
        if MATCH_FIELD:
            STRING_FIELD = MATCH_FIELD.group(1).strip()
            for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
                if EACH_LINE.strip().startswith('FIELDS:'):
                    LIST_LINE[EACH_INDEX_LINE] = f'FIELDS: {STRING_FIELD}'
                    break
            else:
                LIST_LINE.append(f'FIELDS: {STRING_FIELD}')
    INDEX_BASE_EXTENSION = None
    LIST_EXTENSION_BASE = []
    for EACH_INDEX_LINE, EACH_LINE in enumerate(LIST_LINE):
        if EACH_LINE.strip().startswith('EXTENSIONS:'):
            INDEX_BASE_EXTENSION = EACH_INDEX_LINE
            STRING_AFTER = EACH_LINE.split(':', 1)[1].strip()
            if STRING_AFTER and STRING_AFTER != '-':
                LIST_EXTENSION_BASE = [EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_AFTER.split(',') if EACH_EXTENSION.strip()]
            break
    LIST_EXTENSION_ADD = []
    if EACH_REGISTER in MAP_SECTION_EXTENSION:
        MATCH_EXTENSION = re.search(r'^\s*EXTENSIONS:\s*(.+)$', MAP_SECTION_EXTENSION[EACH_REGISTER], flags=re.M)
        if MATCH_EXTENSION:
            LIST_EXTENSION_ADD = [EACH_EXTENSION.strip() for EACH_EXTENSION in MATCH_EXTENSION.group(1).split(',') if EACH_EXTENSION.strip()]
    LIST_EXTENSION_MERGED = sorted(set(LIST_EXTENSION_BASE + LIST_EXTENSION_ADD))
    STRING_EXTENSION_LINE = 'EXTENSIONS: ' + (', '.join(LIST_EXTENSION_MERGED) if LIST_EXTENSION_MERGED else '-')
    if INDEX_BASE_EXTENSION is None:
        LIST_LINE.append(STRING_EXTENSION_LINE)
    else:
        LIST_LINE[INDEX_BASE_EXTENSION] = STRING_EXTENSION_LINE
    if EACH_REGISTER in MAP_SECTION_RULE:
        LIST_RULE_LINE = []
        for EACH_RAW in MAP_SECTION_RULE[EACH_REGISTER].splitlines():
            STRING_RAW = EACH_RAW.strip()
            if STRING_RAW.upper().startswith('RULE '):
                LIST_RULE_LINE.append(STRING_RAW.replace('“', '"').replace('”', '"'))
        if LIST_RULE_LINE:
            NUMBER_RULE_MAX = 0
            for EACH_LINE in LIST_LINE:
                MATCH_RULE = re.match(r'^\s*RULE\s+(\d+)\s*:', EACH_LINE, flags=re.I)
                if MATCH_RULE:
                    NUMBER_RULE_MAX = max(NUMBER_RULE_MAX, int(MATCH_RULE.group(1)))
            NUMBER_NEXT = NUMBER_RULE_MAX + 1
            for EACH_RULE in LIST_RULE_LINE:
                if re.match(r'^\s*RULE\s+\d+\s*:', EACH_RULE, flags=re.I):
                    LIST_LINE.append(EACH_RULE)
                else:
                    LIST_LINE.append(f'RULE {NUMBER_NEXT}: {EACH_RULE}')
                    NUMBER_NEXT += 1
    if EACH_REGISTER not in MAP_SECTION_RULE:
        continue
    LIST_BLOCK_FINAL.append('\n'.join(LIST_LINE).strip())

Path(FILE_OUTPUT).write_text('\n\n'.join(LIST_BLOCK_FINAL), encoding='utf-8')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 7 - CREATE - 7_REGISTER_FINAL.txt - END

0.00 second(s)
====
DONE
====


In [8]:
# 8 - CREATE - 8_REGISTER_TOTAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '5_REGISTER/1_OUTPUT/7_REGISTER_FINAL.txt'
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/8_REGISTER_TOTAL.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_INPUT = FILE_HANDLE.read()

if (not os.path.exists(FILE_OUTPUT)) or os.path.getsize(FILE_OUTPUT) == 0:
    TEXT_OUTPUT = TEXT_INPUT
else:
    with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_PREVIOUS = FILE_HANDLE.read()
    TEXT_COMBINED = TEXT_PREVIOUS.strip() + '\n\n' + TEXT_INPUT.strip()
    REGEX_BLOCK = re.compile(r'(?ms)^\s*REGISTER:\s*([^\n]+)\s*\n.*?(?=^\s*REGISTER:\s*[^\n]+\s*\n|\Z)')
    MAP_BLOCK = {MATCH.group(1).strip(): MATCH.group(0) for MATCH in REGEX_BLOCK.finditer(TEXT_COMBINED)}
    TEXT_OUTPUT = '\n\n'.join(MAP_BLOCK[EACH_NAME].strip() for EACH_NAME in sorted(MAP_BLOCK, key=str.upper)).strip()

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(TEXT_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 8 - CREATE - 8_REGISTER_TOTAL.txt - END

0.00 second(s)
====
DONE
====


In [9]:
# 9 - CREATE - 9_REGISTER_SUMMARY.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from typing import Dict, List, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_OUTPUT = '5_REGISTER/1_OUTPUT/9_REGISTER_SUMMARY.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 9 - CREATE - 9_REGISTER_SUMMARY.txt - END

0.00 second(s)
====
DONE
====
